# Обрезка адаптеров, концов по качеству и длине — лёгкие цепи мыши

Параметры соответствуют обработке тяжёлых цепей ERP003950: явные кандидаты Illumina-адаптеров, обрезка 3′-конца до Q30 и минимальная длина пары 200 нуклеотидов. Последовательности адаптеров взяты из документации Illumina и существующего пайплайна для мыши и эмпирически проверены на этом наборе данных; в статье Kos et al. они не приведены.


In [ ]:
from pathlib import Path
import gzip, json, os, shutil, subprocess, time
REPO=Path.cwd().resolve()
if not (REPO/'notebooks').is_dir(): REPO=REPO.parent
VOLUME=Path(os.environ.get('BCR_VOLUME','/data/user/epishkin'))
if not (VOLUME/'raw').is_dir(): VOLUME=REPO
ENV=Path(os.environ.get('BCR_ENV','/opt/conda/envs/bcr_env'))
if not ENV.is_dir(): ENV=Path('/Users/epishkin/mamba/envs/bcr_env')
RUN='SRR32426580'; RAW=VOLUME/'raw'/'PRJNA1226555'
ROOT=VOLUME/'results'/'PRJNA1226555'/'branches'/'legacy_qtrim_min200'
R1=RAW/f'{RUN}_1.fastq.gz'; R2=RAW/f'{RUN}_2.fastq.gz'
def tool(name):
    p=ENV/'bin'/name
    if p.is_file(): return p
    q=shutil.which(name)
    if q: return Path(q)
    raise FileNotFoundError(name)
def fqcount(path):
    with gzip.open(path,'rt') as h: n=sum(1 for _ in h)
    assert n%4==0,path
    return n//4
def run(cmd,out,err,outputs=(),heartbeat=30):
    out=Path(out); err=Path(err); out.parent.mkdir(parents=True,exist_ok=True)
    started=time.monotonic()
    with out.open('w') as o,err.open('w') as e:
        p=subprocess.Popen([str(x) for x in cmd],stdout=o,stderr=e,text=True)
        print(f'PID={p.pid}',flush=True)
        while p.poll() is None:
            sizes=' '.join(f'{Path(x).name}={Path(x).stat().st_size/1e6:.1f}MB' for x in outputs if Path(x).exists())
            print(f'PID={p.pid} elapsed={(time.monotonic()-started)/60:.1f}min {sizes}',flush=True)
            time.sleep(heartbeat)
    if p.returncode: raise RuntimeError(f'rc={p.returncode}; see {err}')
    print(f'DONE elapsed={(time.monotonic()-started)/60:.1f}min',flush=True)
for p in (R1,R2): assert p.is_file(),p
print('ROOT',ROOT)


In [ ]:
TRIM_QUALITY='0,30'; MIN_LENGTH=200; ADAPTER_TIMES=2; ADAPTER_MIN_OVERLAP=10
ILLUMINA_ADAPTER_R1='AGATCGGAAGAGCGTCGTGTAGGGAAAGAGTGT'
ILLUMINA_ADAPTER_R2='GATCGGAAGAGCACACGTCTGAACTCCAGTCAC'
BASE=ROOT/'trimmed'; FQ=BASE/'fastq'; LOG=BASE/'logs'
shutil.rmtree(BASE,ignore_errors=True); FQ.mkdir(parents=True); LOG.mkdir(parents=True)
t1=FQ/f'{RUN}_1.trim.fastq.gz'; t2=FQ/f'{RUN}_2.trim.fastq.gz'
run([tool('cutadapt'),'--quality-cutoff',TRIM_QUALITY,'-m',str(MIN_LENGTH),'--times',str(ADAPTER_TIMES),'-O',str(ADAPTER_MIN_OVERLAP),'--compression-level','1','-a',ILLUMINA_ADAPTER_R1,'-A',ILLUMINA_ADAPTER_R2,'--json',LOG/f'{RUN}.cutadapt.json','-o',t1,'-p',t2,R1,R2],LOG/f'{RUN}.stdout.log',LOG/f'{RUN}.stderr.log',[t1,t2])
assert fqcount(t1)==fqcount(t2)
summary={'raw_pairs':fqcount(R1),'trimmed_pairs':fqcount(t1),'retention':fqcount(t1)/fqcount(R1)}
(ROOT/'trimmed'/'trim_summary.json').write_text(json.dumps(summary,indent=2))
print(summary)
